### GridBatch and JaggedTensor
There are two fundamental classes in fVDB you will encounter frequently: `GridBatch` and `JaggedTensor`.

```python
fvdb.GridBatch
fvdb.JaggedTensor
```

#### GridBatch

A `GridBatch` is an indexing structure which maps the 3D $ijk$ coordinates of a set of **sparse** grids to integer offsets which can be used to look up elements in a data tensor that correspond with each voxel.

This mapping only exists for $ijk$ coordinates which are **active** in the space of a **sparse**, 3D grid.

&#x1F4A1; We call these 3D grids **sparse** because of this arbitrary nature to the topology of **active** voxels in the grid compared to **dense** grids where all voxels exist inside some regular extents in each dimension.

The figure below illustrates this $ijk$ mapping process for a `GridBatch` containing only a single grid.


<center>

<img src="img/gridbatch.svg"  alt="Image Index Grid" width="800"/>

</center>

In practice, `GridBatch` is an ordered collection of 1 or more of these 3D grids.  

&#x1F529;	At the level of technical implementation, these 3D grids are [**NanoVDB**](https://academysoftwarefoundation.github.io/openvdb/NanoVDB_MainPage.html) grids of the special `IndexGrid` type which only stores a unique **index** integer value at each active voxel location.  This **index** is an offset into some external data array, a tensor, (one that is not contained in the `IndexGrid`/`GridBatch` classes) where contiguous array members correspond to spatially nearby voxels.  `IndexGrid` will allow us to reference into this 'sidecar' tensor of data given a spatial $ijk$ set of coordinates.

Each grid member in a `GridBatch` can have different topologies, different numbers of active voxels and different voxel dimensions and origins per-grid.

<center>

<img src="img/gridbatch_concept.svg"  alt="Image Index Grid" width="900"/>

</center>

##### Images as 2D GridBatch


To help explain these concepts, let's consider how we might treat image data with this framework (an image can be thought of as a dense 2D grid after all).  

If an image is a 2D grid of pixels, we could imagine the position of each pixel could be expressed as $i,j$ coordinates.  For an RGB image, each pixel would contain 3 associated values $(R,G,B)$.

In this way, we could decompose an image into:

1.  an `IndexGrid` of $i,j$ coordinates
2.  a tensor of size $[NumberPixels, 3]$ (the flattened RGB elements for our data tensor)

The $i,j$ coordinates can be used with the `IndexGrid` to index into the list of RGB values to retrieve the RGB element for each pixel.  In our nomenclature, we'd say our data has element size 3 (RGB) and the number of elements in our data tensor is equal to the number of pixels in the image (in the illustration below the number of elements is 16).

<center>

<img src="img/image_index_grid_diagram.svg"  alt="Image Index Grid" width="800"/>

</center>

If we had a batch of *many* images, each of different sizes, we can imagine constructing a `GridBatch` as an ordered set of their `IndexGrid`s.

All the RGB elements would go into a sidecar data tensor where each array of data would have to be of a different length corresponding to the image size.  We call this list of different-length element data a **jagged tensor**.

<center>

<img src="img/grid_batch_diagram.svg"  alt="Grid Batch" width="600"/>

</center>

This is the essence of the relationship between `GridBatch` and `JaggedTensor` in fVDB.  But more on `JaggedTensor` later…

Lastly, it is important to know that each grid in the `GridBatch` will be on the same device and processed together by operators in mini-batch-like fashion.  

Let's put together our first `GridBatch` to see how it works.

In [10]:
# import the usual suspects and fvdb
import numpy as np
import torch
import fvdb

We will make a `GridBatch` of 8 grids, each with a different number of active voxels and all of those voxels' positions will be chosen randomly.  Further, each grid will have a randomly chosen origin in the 3D world space and a randomly chosen voxel size.

In [11]:
batch_size = 8
# Randomly generate different numbers of voxels we desire in each grid in our batch
num_voxels_per_grid = [np.random.randint(100, 1_000) for _ in range(batch_size)]

# A list of randomly generated 3D indices for each grid in our batch in the range [-512, 512]
ijks = [torch.randint(-512, 512, (num_voxels_per_grid[i], 3), device="cuda") for i in range(batch_size)]

# ijks is currently a list of tensors, we need to convert it to a JaggedTensor
jagged_ijks = fvdb.JaggedTensor(ijks)

# Create an fvdb.GridBatch from the list of indices!!
grid_batch = fvdb.GridBatch.from_ijk(
    jagged_ijks,  # We'll explain JaggedTensor in a moment…
    # Random, different voxel sizes for each grid in our batch
    voxel_sizes=[np.random.rand(3).tolist() for _ in range(batch_size)],
    # Random, different grid origins for each grid in our batch
    origins=[np.random.rand(3).tolist() for _ in range(batch_size)],
)

There are many more convenient ways that fVDB provides to create a `GridBatch` besides building from lists of coordinate indexes such as building a `GridBatch` from **worldspace pointclouds, meshes or dense tensors**.

&#x1F4A1; The fVDB documentation has more useful examples for these cases using functions like `GridBatch.from_points`, `GridBatch.from_dense` and `GridBatch.from_mesh`.

In [12]:
# This grid will be on the GPU because the `ijks` were on that device
assert(grid_batch.device == ijks[0].device == torch.device('cuda:0'))

Each member of the batch has different voxel size dimensions, a different origin in space and different number of voxels

In [13]:
for i in range(grid_batch.grid_count):
    print(f"""Grid {i} has {grid_batch.num_voxels_at(i)} voxels,
        voxel size of {grid_batch.voxel_size_at(i).tolist()}
        and an origin of {grid_batch.origin_at(i).tolist()}""")

Grid 0 has 342 voxels,
        voxel size of [0.7894321084022522, 0.763862669467926, 0.8576123118400574]
        and an origin of [0.7543044686317444, 0.4581454396247864, 0.44815802574157715]
Grid 1 has 215 voxels,
        voxel size of [0.2675798237323761, 0.3951326012611389, 0.5802198052406311]
        and an origin of [0.7356592416763306, 0.34672775864601135, 0.750876247882843]
Grid 2 has 565 voxels,
        voxel size of [0.3130077123641968, 0.7809085249900818, 0.3005487024784088]
        and an origin of [0.557346522808075, 0.8767868876457214, 0.27118736505508423]
Grid 3 has 841 voxels,
        voxel size of [0.4236169457435608, 0.10990063846111298, 0.1318466067314148]
        and an origin of [0.17290891706943512, 0.5779466032981873, 0.7602676749229431]
Grid 4 has 601 voxels,
        voxel size of [0.4993608295917511, 0.19063852727413177, 0.1742287576198578]
        and an origin of [0.9692448377609253, 0.36057722568511963, 0.040953148156404495]
Grid 5 has 278 voxels,
        vox

Let's examine some of the ways we can retrieve indices from a `GridBatch` based on $ijk$ coordinates.

In [14]:
# Let's retrieve a random ijk coordinate from each of the lists we used to make the grids
ijk_queries = fvdb.JaggedTensor([ijks[n][np.random.randint(len(ijks[n]))][None,:] for n  in range(grid_batch.grid_count)])

# Use the GridBatch to get indices into the sidecar feature array from the `ijk` coordinate in each grid
feature_indices = grid_batch.ijk_to_index(ijk_queries)
world_positions = grid_batch.voxel_to_world(ijk_queries.float())
for i, (ijk, world_p, i_f) in enumerate(zip(ijk_queries, world_positions, feature_indices)):
    print(f"Grid {i}, feature Index at ijk {ijk.jdata.tolist()}, world-space {world_p.jdata.tolist()} : {i_f.jdata.item()}")

# NOTE: This GridBatch (Batch of IndexGrids) just expresses the topology of the grids and can be used to reference a sidecar flat array of features but we won't create this sidecar in this example…
# We can get the index into this hypothetical sidecar feature array with any `ijk` coordinate (if we ask for an `ijk` not in the grid, -1 is given as the index)

Grid 0, feature Index at ijk [[139, -148, -474]], world-space [[110.48536682128906, -112.5935287475586, -406.0600891113281]] : 186
Grid 1, feature Index at ijk [[-249, 471, -292]], world-space [[-65.89171600341797, 186.4541778564453, -168.67330932617188]] : 79
Grid 2, feature Index at ijk [[357, 199, -312]], world-space [[112.30110168457031, 156.277587890625, -93.5]] : 468
Grid 3, feature Index at ijk [[275, -245, 170]], world-space [[116.66757202148438, -26.34771156311035, 23.174192428588867]] : 582
Grid 4, feature Index at ijk [[-203, 107, 464]], world-space [[-100.40100860595703, 20.758899688720703, 80.88309478759766]] : 287
Grid 5, feature Index at ijk [[430, 450, -214]], world-space [[177.49781799316406, 164.4188232421875, -57.52029037475586]] : 236
Grid 6, feature Index at ijk [[-160, 236, -370]], world-space [[-82.57447052001953, 217.56520080566406, -294.4183654785156]] : 67
Grid 7, feature Index at ijk [[331, 98, 139]], world-space [[221.54885864257812, 58.77158737182617, 102.6

#### JaggedTensor

`JaggedTensor` is the supporting element data (i.e. the 'sidecar' of data) that is paired with a `GridBatch`.  

You can think of `JaggedTensor` as consisting of an ordered list of PyTorch Tensors, one for each grid in the `GridBatch`.  Same as `GridBatch`, the Tensors are all on the same device and processed together in a mini-batch. 

In [15]:
# Here we create a list of Tensors, one for each grid in the batch with its number of active voxels and 5 random features per-voxel
list_of_features = [torch.randn(int(grid_batch.num_voxels[i]), 5, device=grid_batch.device) for i in range(grid_batch.grid_count)]
# Now we make a JaggedTensor out of this list of heterogeneous Tensors
features  = fvdb.JaggedTensor(list_of_features)

In the `JaggedTensor` above, you can see how we constructed it with a list of heterogeneously sized Tensors whose shapes were of the form:

$$[ [B1, E], [B2, E], [B3, E], …] ]$$

where the value of $Bn$ would be the number of active voxels in each grid of the `GridBatch` and $E$ is the number of elements (5 was chosen in this case).

&#x1F4A1; Note how each Tensor element in our `JaggedTensor` can have different numbers of active voxels (similar to `GridBatch`) but the same number of per-voxel elements.  This is distinctly different from the classic representation of 3D data in a PyTorch `Tensor` which is usually a homogeneously shaped Tensor of shape $[N, C, H, W, D]$ where $N$ would be the number of "grids" in our batch, the $C$ *channels* are equivalent to the size of the $E$ elements, and $H, W, D$ are the 3 index dimensions of a **dense** grid.

We can also directly use a `GridBatch` to derive a `JaggedTensor` to match the `GridBatch`'s specific sizing.

This more concise line of code has the same effect as the code above:

In [16]:
# From a GridBatch, a matching JaggedTensor can be created from a Tensor of shape `(total_voxels, feature_dim)`
features = grid_batch.jagged_like(torch.randn(grid_batch.total_voxels, 5, device=grid_batch.device))

Now that we have a `GridBatch` and a `JaggedTensor` of elements that correspond to the batch of grids, we could use the $ijk$ offsets we can obtain from the `GridBatch` to index into the `JaggedTensor` to retrieve the data for the voxels at those $ijk$'s.

In [17]:
features[feature_indices].jdata

tensor([[ 0.1783,  0.1195, -0.7551, -1.3581, -1.3193],
        [-0.1905,  0.7448, -0.7126,  1.3617, -1.7972],
        [ 0.2396,  0.5921, -1.2450,  0.1734,  2.4539],
        [-0.7335, -0.1038, -0.7046,  0.1622, -0.2558],
        [-0.9191,  0.8270, -0.0693, -1.4802, -0.1048],
        [-1.8124, -0.2121, -0.3573, -0.2792, -0.7942],
        [-0.2395, -1.2487,  0.0604, -0.1358,  0.5021],
        [ 0.7842,  0.0194,  0.7410,  0.1877, -0.6972]], device='cuda:0')

But beyond just looking up the elements directly, we can use higher-level ƒVDB functionality like sampling what the values are at a given worldspace position.

Let's get the data values at the worldspace positions used earlier in the lesson:

In [18]:
sampled_features = grid_batch.sample_trilinear(world_positions, features)


for xyz, value in zip(world_positions.jdata, sampled_features.jdata):
    print(f"World position {[round(num, 1) for num in xyz.tolist()]} \n\thas the values: {[round(num, 3) for num in value.tolist()]}")

World position [110.5, -112.6, -406.1] 
	has the values: [0.178, 0.12, -0.755, -1.358, -1.319]
World position [-65.9, 186.5, -168.7] 
	has the values: [-0.191, 0.745, -0.713, 1.362, -1.797]
World position [112.3, 156.3, -93.5] 
	has the values: [0.24, 0.592, -1.245, 0.173, 2.454]
World position [116.7, -26.3, 23.2] 
	has the values: [-0.734, -0.104, -0.705, 0.162, -0.256]
World position [-100.4, 20.8, 80.9] 
	has the values: [-0.919, 0.827, -0.069, -1.48, -0.105]
World position [177.5, 164.4, -57.5] 
	has the values: [-1.812, -0.212, -0.357, -0.279, -0.794]
World position [-82.6, 217.6, -294.4] 
	has the values: [-0.239, -1.249, 0.06, -0.136, 0.502]
World position [221.5, 58.8, 102.7] 
	has the values: [0.784, 0.019, 0.741, 0.188, -0.697]


### JaggedTensor Implementation Details


Internally, `JaggedTensor` does not represent the list of features as a list of differently sized PyTorch tensors, but instead stores a single tensor of the element data accompanied by a special index structure that allows us to access the data for each grid in the `GridBatch`.  

The `jdata` attribute of a `JaggedTensor` contains all the element values in this single list.  `jdata` is a `Tensor` of shape $[N, E]$ where $N$ is the total number of active voxels in the batch and $E$ is the element size.  

`jdata`'s shape would be equivalent to the result of concatenating the list of heterogeneously sized Tensors, mentioned above, along their first axis into a single Tensor whose shape would be $[B1+B2+B3…+Bn, E]$.

<center>

<img src = "img/jdata.jpg" width=1200 alt="jdata">

</center>

In [19]:
print(f"size of element data for the entire GridBatch: {features.jdata.shape}")

size of element data for the entire GridBatch: torch.Size([3901, 5])


To determine which grid each element belongs to, `JaggedTensor` contains indexing information in its `jidx` attribute.

`jidx` is a `Tensor` of shape $[N]$ where $N$ is again the total number of active voxels across the batch.  Each member of `jidx` is an integer that tells us which grid in the `GridBatch` the corresponding element in `jdata` belongs to.  The grid membership in `jidx` is ordered starting from 0 and members of the same batch are contiguous.

<center>

<img src = "img/jidx.jpg" width=1200 alt="jdata">

</center>

In [20]:
print(f"per-element membership for each voxel in the GridBatch: {features.jidx}")
print(f"\nthe size of the elements of the 4th grid in this batch: {features.jdata[features.jidx==3].shape}")

per-element membership for each voxel in the GridBatch: tensor([0, 0, 0,  ..., 7, 7, 7], device='cuda:0', dtype=torch.int32)

the size of the elements of the 4th grid in this batch: torch.Size([841, 5])


Additionally, `JaggedTensor` has a `joffsets` attribute that can also be used to index into `jdata` to get the element data for each grid in the batch.

The `joffsets` attribute is a `Tensor` of shape $[B]$, where $B$ is the number of grids in the batch.  `joffset`'s values are the start offsets into `jdata` that corresponds to each grid in the batch.  This is essentially the same information that can be found in `jidx` but expressed in a different form.

<center>

<img src = "img/joffsets.jpg" width=1200 alt="jdata">

</center>

In [21]:
print("per-grid offsets into the data:")
print(features.joffsets)
print(f"\nthe size of the elements of the 4th grid in this batch: {features.jdata[features.joffsets[3]:features.joffsets[3+1]].shape}")

per-grid offsets into the data:
tensor([   0,  342,  557, 1122, 1963, 2564, 2842, 3073, 3901], device='cuda:0')

the size of the elements of the 4th grid in this batch: torch.Size([841, 5])


The element data stored in a `JaggedTensor` can be of any type that PyTorch supports, including float, float64, float16, bfloat16, int, etc., and the elements can have any arbitrary size.  In fact, elements can contain multi-dimensional tensor data.

For instance, there could be a `JaggedTensor` with 1 float that represents a signed distance field in each grid, or float elements of size 3 that represent an RGB color in each voxel of the grids, or elements of shape (3,3) representing 3x3 matrices, or float elements of length 192 that represent a learned feature vector of each voxel in each grid.

In [22]:
# A single scalar element per voxel
features = grid_batch.jagged_like(torch.randn(grid_batch.total_voxels, 1, dtype=torch.float, device=grid_batch.device))

# Cast to a double
features = features.double()

# A JaggedTensor of 3x3 matrices for element data
features = grid_batch.jagged_like(torch.randn(grid_batch.total_voxels, 3, 3, dtype=torch.float, device=grid_batch.device))

# A JaggedTensor of 192 float elements
features = grid_batch.jagged_like(torch.randn(grid_batch.total_voxels, 192, dtype=torch.float, device=grid_batch.device))